# Optimización de Antenas 5G mediante Recocido Simulado

En este notebook implementamos el algoritmo de Recocido Simulado para ajustar las potencias de transmisión de 100 antenas 5G. El objetivo es maximizar la cobertura en una zona rural sintética, aplicando una penalización cuando las señales se solapan (interferencia).

Nota: Para asegurar que el algoritmo corra de forma autónoma, incluimos temporalmente la lógica base de la grilla y atenuación de la señal en este mismo archivo.

In [ ]:
using Random
using Statistics
using Printf
using Plots
using DelimitedFiles
using DataFrames
using CSV

mkpath("../analytics/output")


## 1. Definición del modelo físico

Planteamos un área de 10x10 km. La atenuación de la señal se calcula con un modelo simplificado de pérdida en espacio libre para la banda de 3500 MHz.

In [ ]:
# Parámetros físicos base
freq_mhz = 3500.0
p_max_w = 1.0
umbral_dbm = -80.0
h_antena_m = 30.0
h_persona_m = 1.5
n_antenas = 100

# Malla de antenas (10x10 km, separación de 1 km)

pos_antenas = []
for x in 0.5:1.0:9.5
    for y in 0.5:1.0:9.5
        push!(pos_antenas, (x, y))
    end
end


# Puntos de demanda poblacional (separación de 200 m)
poblacion = []
for x in 0.1:0.2:9.9
    for y in 0.1:0.2:9.9
        push!(poblacion, (x, y))
    end
end

function hay_cobertura(persona, antena, potencia_w)
    if potencia_w == 0.0
        return false
    end
    
    dx_km = persona[1] - antena[1]
    dy_km = persona[2] - antena[2]
    dz_km = (h_antena_m - h_persona_m) / 1000.0
    dist_km = sqrt(dx_km^2 + dy_km^2 + dz_km^2)
    
    perdida = 32.4 + 20 * log10(freq_mhz) + 20 * log10(dist_km)
    tx_dbm = 10 * log10(potencia_w) + 30.0
    rx_dbm = tx_dbm - perdida
    
    return rx_dbm >= umbral_dbm
end

function evaluar_fitness(potencias_w::Vector{Float64})
    cubiertos = 0
    for p in poblacion
        for (i, a) in enumerate(pos_antenas)
            if hay_cobertura(p, a, potencias_w[i])
                cubiertos += 1
                break 
            end
        end
    end
    
    # Penalización temporal (λ=0.5) hasta integrar el módulo oficial
    lambda_interf = 0.5
    interferencia = sum(potencias_w) * lambda_interf
    return Float64(cubiertos) - interferencia
end

## 2. Algoritmo de Recocido Simulado

Para buscar el vector óptimo de potencias aplicamos:
- **Vecindario:** Alteramos entre 1 y 5 potencias al azar con ruido gaussiano, topando los valores en $[0, P_{\max}]$.
- **Enfriamiento:** Esquema geométrico clásico.
- **Temperatura inicial:** Se calibra con una muestra previa para lograr una aceptación del 80% al principio.

In [ ]:
function vecino_potencias(pot_w::Vector{Float64}, p_max_w::Float64; k_max::Int=5, sigma_pct::Float64=0.1)
    vecino_w = copy(pot_w)
    k = rand(1:k_max) 
    indices = randperm(length(pot_w))[1:k] 
    ruido_w = sigma_pct * p_max_w

    for i in indices
        vecino_w[i] += randn() * ruido_w
        # Restringimos al dominio factible [0, P_max]
        vecino_w[i] = clamp(vecino_w[i], 0.0, p_max_w)
    end

    return vecino_w
end

In [ ]:
function calibrar_t0(fitness_fn, pot_w::Vector{Float64}, p_max_w::Float64; n_muestras::Int=500, target_acc::Float64=0.8)
    fit_inicial = fitness_fn(pot_w)
    deltas_negativos = Float64[]

    for _ in 1:n_muestras
        v_w = vecino_potencias(pot_w, p_max_w)
        fit_vecino = fitness_fn(v_w)
        delta = fit_vecino - fit_inicial
        if delta < 0
            push!(deltas_negativos, abs(delta))
        end
    end

    if isempty(deltas_negativos)
        return 1.0 # Fallback si no hay empeoramientos en la muestra
    end

    delta_promedio = mean(deltas_negativos)
    return -delta_promedio / log(target_acc)
end

In [ ]:
struct ResultadoSA
    mejor_potencia_w::Vector{Float64}
    mejor_fit::Float64
    hist_mejor_fit::Vector{Float64}
    hist_temp::Vector{Float64}
    tiempo_s::Float64
    semilla::Int
end

In [ ]:
function run_sa(fitness_fn, p_max_w::Float64, n_antenas::Int;
                n_iters::Int=50_000,
                alpha_ratio::Float64=1e-4,
                semilla::Int=42)

    Random.seed!(semilla)

    pot_w = rand(n_antenas) .* p_max_w
    fit_actual = fitness_fn(pot_w)

    mejor_pot_w = copy(pot_w)
    mejor_fit = fit_actual

    t0 = calibrar_t0(fitness_fn, pot_w, p_max_w)
    temp = t0
    alpha = alpha_ratio ^ (1.0 / n_iters)

    hist_mejor_fit = zeros(n_iters)
    hist_temp = zeros(n_iters)

    t_inicio = time()

    for i in 1:n_iters
        v_w = vecino_potencias(pot_w, p_max_w)
        fit_v = fitness_fn(v_w)

        delta = fit_v - fit_actual

        # Criterio de aceptación de Metrópolis (Maximización)
        if delta >= 0 || rand() < exp(delta / temp)
            pot_w = v_w
            fit_actual = fit_v

            if fit_actual > mejor_fit
                mejor_pot_w = copy(pot_w)
                mejor_fit = fit_actual
            end
        end

        hist_mejor_fit[i] = mejor_fit
        hist_temp[i] = temp

        temp *= alpha
    end

    tiempo_s = time() - t_inicio

    return ResultadoSA(mejor_pot_w, mejor_fit, hist_mejor_fit, hist_temp, tiempo_s, semilla)
end

## 3. Pruebas y Resultados

Corremos el algoritmo 5 veces con distintas semillas. Dejamos 50,000 iteraciones como límite base para luego poder compararlo directamente con los algoritmos poblacionales.

In [ ]:
semillas = [42, 123, 256, 789, 1024]
resultados = ResultadoSA[]

for s in semillas
    res = run_sa(evaluar_fitness, p_max_w, n_antenas, n_iters=50_000, semilla=s)
    push!(resultados, res)
end

In [ ]:
p1 = plot(title="Evolución del Fitness (5 corridas)", xlabel="Iteración", ylabel="Mejor Fitness", legend=:bottomright, grid=true, size=(800, 450))
colores = [:blue, :red, :green, :orange, :purple]
for (i, res) in enumerate(resultados)
    plot!(p1, res.hist_mejor_fit, label="Semilla $(res.semilla)", color=colores[i], lw=1.5)
end
display(p1)
savefig(p1, "../analytics/output/sa_convergencia_semillas.png")

In [ ]:
mejores_fits = [r.mejor_fit for r in resultados]
tiempos = [r.tiempo_s for r in resultados]

for r in resultados
    @printf("Semilla %d: Fitness = %.4f | Tiempo = %.2fs\n", r.semilla, r.mejor_fit, r.tiempo_s)
end

@printf("\nPromedio: %.4f ± %.4f\n", mean(mejores_fits), std(mejores_fits))
idx_mejor = argmax(mejores_fits)
@printf("Mejor global: Semilla %d con %.4f\n", resultados[idx_mejor].semilla, mejores_fits[idx_mejor])

In [ ]:
mejor_res = resultados[idx_mejor]

# Guardar mejor solucion con coordenadas para compatibilidad con Yosue
ids = 1:n_antenas
pos_x_arr = [p[1] for p in pos_antenas]
pos_y_arr = [p[2] for p in pos_antenas]

df_solucion = DataFrame(
    ID_Antena = ids,
    Pos_X = pos_x_arr,
    Pos_Y = pos_y_arr,
    Potencia_W = mejor_res.mejor_potencia_w
)
CSV.write("../analytics/output/potencias_por_lambda.csv", df_solucion)

# Guardar el historial de la mejor solucion para Yosue
open("../analytics/output/historial_SA_mejor.csv", "w") do io
    println(io, "Iteracion,Fitness")
    for (i, fit) in enumerate(mejor_res.hist_mejor_fit)
        println(io, "$i,$fit")
    end
end

# Guardar matriz de convergencia de todas las semillas (Opcional)
hist_mat = hcat([r.hist_mejor_fit for r in resultados]...)
open("../analytics/output/sa_datos_convergencia.csv", "w") do io
    println(io, join(["Semilla_$(r.semilla)" for r in resultados], ","))
    writedlm(io, hist_mat, ',')
end


## Conclusiones del modelo

- El cálculo dinámico de la temperatura resulta clave para mantener una exploración útil al inicio, independientemente de los pesos que le pongamos a la penalización.
- La dispersión (desviación estándar) entre las 5 corridas nos indica la presencia de óptimos locales en la grilla.